In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("MY_GITHUB_TOKEN")
os.environ["GIT_TERMINAL_PROMPT"] = "0"          # fail fast if auth fails, never prompt

# 2) Fresh clone into the Kaggle working dir (NOT /content — that's Colab)
!rm -rf sports_prediction_model
!git clone -q https://github.com/andrewkemmer/sports_prediction_model.git sports_prediction_model

# 3) Install every dependency the pipeline imports (backend, not the Streamlit frontend)
!pip install -q nflreadpy gitpython polars scikit-learn \
    lightgbm xgboost pandas numpy joblib requests
print("setup done")



import os

# --- OPTIONAL overrides (omit everything for a normal run) ---

# Slate target season: the season the board predicts and the sealed-2025 gate
# holds out. Omit = current calendar year; set only for a backfill.
os.environ["NFL_START_SEASON"] = "2019"   # <- your start selector
os.environ["NFL_END_SEASON"]   = "2025"   # <- your end selector
os.environ["NFL_SLATE_SEASON"] = "2026"   # the board's target season

# NOTE: unlike MLB there is no start/end window or full-repull knob — NFL Phase 1
# always rebuilds the decided frame from the full 2019-2025 history.
# (Dry-run without pushing: add "--no-push" to the command in the next cell.)

print("run options set")




import os
import subprocess

repo = "/kaggle/working/sports_prediction_model"
cmd = ["python", "nfl-backend/backend/master_pipeline.py"]

result = subprocess.run(cmd, cwd=repo, env=os.environ.copy(), capture_output=False)

# The pipeline's Phase 4 already pushes data_delivery/ to GitHub itself.
# Fail loudly if it didn't complete — Kaggle marks the run failed.
if result.returncode != 0:
    raise SystemExit(f"Pipeline failed with exit code {result.returncode}")
print("Pipeline completed — artifacts pushed to GitHub by Phase 4 sync.")



import os
import subprocess

repo = "/kaggle/working/sports_prediction_model"

# Confirm main moved: fetch and compare HEAD to the pre-run HEAD
check = subprocess.run(
    ["git", "log", "--oneline", "-1"],
    cwd=repo, capture_output=True, text=True,
)
print("Repo HEAD after run:", check.stdout.strip())

# Sanity: newest dated artifact on main
ls = subprocess.run(
    ["git", "ls-tree", "-r", "--name-only", "origin/main"],
    cwd=repo, capture_output=True, text=True,
)
datelated = [f for f in ls.stdout.splitlines() if "nfl-backend/data_delivery/" in f and "_202" in f]
print("Latest dated artifacts:", sorted(datelated)[-3:] if datelated else "none found")
